# Orography Processing Tutorial

This notebook prepares the DEM terrain variables required by the ATLAS solar workflow.

It produces country level NetCDF files for:

1. ERA5 Land orography
2. ERA5 Land aspect
3. GLO 90 orography
4. GLO 90 aspect

All paths are relative and anonymous. The only country specific folder used by default is:

```python
../DEMdata/{country}/
```

The notebook is organised as a tutorial. Users should edit only the **User parameters** cell, then run the remaining cells in order.

## Step 1. User parameters
The first dataset required is the ERA5-Land geopotential, a global, time-invariant land parameter provided by ECMWF as part of the ERA5-Land dataset.

The geopotential field represents the geopotential value at the terrain surface and is provided at a spatial resolution of 0.1° × 0.1°.

Additional documentation is available at:

https://confluence.ecmwf.int/display/CKB/ERA5-Land%3A+data+documentation

The dataset can be downloaded directly from:

https://confluence.ecmwf.int/download/attachments/140385202/geo_1279l4_0.1x0.1.grib2_v4_unpack.nc?version=1&modificationDate=1591983422003&api=v2

Place the downloaded NetCDF file inside this directory.
Expected input files inside `./DEMdata/{country}/`:

```text
geo_1279l4_0.1x0.1.grib2_v4_unpack.nc
```

The Copernicus GLO-90 DEM is **not included** in the repository and must be downloaded separately from:

https://portal.opentopography.org/raster?opentopoID=OTSDEM.032021.4326.1

To download the DEM:

1. Open the OpenTopography portal.
2. Select the grid tile(s) corresponding to the country of interest.
3. Keep **GeoTIFF** as the output format.
4. Under **Raster Visualization**, select **Aspect**.
5. Download the resulting GeoTIFF file(s).

If the GLO-90 DEM is downloaded as multiple tiles, list their paths in `glo90_tile_paths`. The notebook will merge them automatically before processing.

### Boundary data

This notebook uses only Natural Earth country boundaries. Do not set a local boundary file. The country geometry is selected automatically from `natural_earth_path` using the selected `country`. Source: https://www.naturalearthdata.com/

In [1]:
from pathlib import Path

# Country name used in file names and folder names.
# Use lowercase names, for example: "argentina", "peru", "chile".
country = "peru"

# Main DEM folder for this country.
dem_path = Path(f"../DEMdata/{country}")
input_path = dem_path
output_path = dem_path
output_path.mkdir(parents=True, exist_ok=True)

# ERA5 Land geopotential input.
# This file is used to derive the ERA5 Land orography.
era5_geopotential_path = Path("../DEMdata/geo_1279l4_0.1x0.1.grib2_v4_unpack.nc")

# GLO 90 DEM input.

# If the DEM was downloaded in multiple parts, list them here.
# Leave this list empty if you already have a single complete DEM file.
glo90_tile_paths = []
# EXAMPLE
#if country == "argentina":
glo90_tile_paths = [
    input_path / f"glo90_orography_{country}_north.tif",
    input_path / f"glo90_orography_{country}_south.tif",
]


# If you already have one complete country file, keep this path.
#else:
glo90_orography_path = input_path / f"glo90_orography_{country}.tif" #Name of the file that you should download from Open Topography


# Country boundary settings.
# Choose one of the following options:
# "local_dissolve": read a local shapefile and dissolve all features into one country geometry.
# "local_filter": read a local shapefile and select features using one attribute column.
# "natural_earth": read a Natural Earth global country shapefile and select the country by name.
#boundary_source = "local_dissolve"

# Recommended for Chile when using REGIONES_v1.shp.
# The shapefile contains one feature per region. The notebook dissolves all regions into one country geometry.

# Optional settings used only when boundary_source = "local_filter".
# Example: local_boundary_column = "REGION" and local_boundary_values = ["Coquimbo", "Valparaíso"]
# Leave local_boundary_values as None to keep all records.
#local_boundary_column = None
#local_boundary_values = None

# Natural Earth fallback settings, used only when boundary_source = "natural_earth".
natural_earth_path = Path("../world_map/ne_50m_admin_0_countries.shp")
natural_earth_country_column = "NAME_EN"
natural_earth_country_name = None  # Leave as None to use country.capitalize().

# Select which products to generate.
run_era5_land = True
run_glo90 = True

# If True, existing outputs are overwritten.
overwrite = True

## Step 2. Imports

Run this cell once. The notebook requires a geospatial Python environment with `geopandas`, `rioxarray`, `rasterio`, `xarray` and `xdem`.

In [2]:
import os
import math
import warnings

import geopandas as gpd
import numpy as np
import rasterio
import rioxarray
import xarray as xr
import xdem

from rasterio.merge import merge

warnings.filterwarnings("ignore", category=FutureWarning)

## Step 3. Helper functions

These functions standardise coordinate names, save NetCDF files with compression, load the country boundary, merge DEM tiles if needed, and clip raster data to the country geometry.

The boundary loader supports the loaded country level shapefile, already containing the national geometry.

All geometries are reprojected to EPSG:4326 before being used for the crop.


In [3]:
def rename_coordinates(ds):
    """Rename common spatial coordinate names to longitude and latitude."""
    rename_map = {}

    if "x" in ds.coords:
        rename_map["x"] = "longitude"
    if "y" in ds.coords:
        rename_map["y"] = "latitude"
    if "lon" in ds.coords:
        rename_map["lon"] = "longitude"
    if "lat" in ds.coords:
        rename_map["lat"] = "latitude"

    if rename_map:
        ds = ds.rename(rename_map)

    return ds


def drop_auxiliary_coordinates(ds):
    """Remove auxiliary coordinates that are not needed in the final NetCDF files."""
    for coord in ["spatial_ref", "band"]:
        if coord in ds.coords:
            ds = ds.drop_vars(coord)

    return ds.squeeze(drop=True)


def roll_longitudes(ds):
    """Convert longitudes from 0 to 360 into -180 to 180 when needed."""
    if "longitude" in ds.coords and not bool((ds.longitude < 0).any()):
        ds = ds.assign_coords(longitude=((ds.longitude + 180) % 360) - 180)
        ds = ds.sortby("longitude")

    return ds


def prepare_spatial_dataset(ds):
    """Prepare a raster dataset for clipping with rioxarray."""
    ds = rename_coordinates(ds)
    ds = drop_auxiliary_coordinates(ds)
    ds = roll_longitudes(ds)

    ds["latitude"] = ds.latitude.astype("float32")
    ds["longitude"] = ds.longitude.astype("float32")

    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    ds = ds.rio.write_crs("EPSG:4326", inplace=False)

    return ds


def save_netcdf_compressed(ds, path, complevel=4, use_float32=True):
    """Save a Dataset to NetCDF using compression."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    encoding = {}

    for var in ds.data_vars:
        var_encoding = {
            "zlib": True,
            "complevel": complevel,
            "shuffle": True,
        }

        if use_float32 and np.issubdtype(ds[var].dtype, np.floating):
            var_encoding["dtype"] = "float32"

        encoding[var] = var_encoding

    ds.to_netcdf(path, engine="netcdf4", encoding=encoding)


def _read_vector_file(path):
    """Read a vector file and check that it contains at least one geometry."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Boundary file not found: {path}")

    gdf = gpd.read_file(path)

    if gdf.empty:
        raise ValueError(f"Boundary file contains no features: {path}")

    if gdf.crs is None:
        raise ValueError(
            f"Boundary file has no CRS: {path}. Define the CRS before running this notebook."
        )

    return gdf


def _filter_geodataframe(gdf, column=None, values=None):
    """Optionally filter a GeoDataFrame using an attribute column."""
    if column is None or values is None:
        return gdf

    if column not in gdf.columns:
        raise ValueError(
            f"Column '{column}' was not found in the boundary file. Available columns are: {list(gdf.columns)}"
        )

    if isinstance(values, (str, int, float)):
        values = [values]

    filtered = gdf[gdf[column].isin(values)]

    if filtered.empty:
        raise ValueError(f"No geometries found in column '{column}' for values: {values}")

    return filtered


def dissolve_to_single_geometry(gdf):
    """Dissolve all features into one single geometry in EPSG:4326."""
    gdf = gdf.to_crs("EPSG:4326")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    if gdf.empty:
        raise ValueError("No valid geometries were found after removing empty geometries.")

    dissolved = gdf.dissolve()
    dissolved = dissolved.explode(index_parts=False).reset_index(drop=True)

    # Return a GeoSeries because rioxarray.rio.clip accepts a GeoSeries or a list of geometries.
    return dissolved.geometry


def load_country_geometry(
    country,
    natural_earth_path,
    natural_earth_country_column="NAME_EN",
    natural_earth_country_name=None,
):
    """Load the selected country geometry from Natural Earth only."""
    natural_earth_path = Path(natural_earth_path)

    if not natural_earth_path.exists():
        raise FileNotFoundError(
            f"Natural Earth shapefile not found: {natural_earth_path}. "
            "Please update natural_earth_path in the User parameters cell."
        )

    gdf = gpd.read_file(natural_earth_path)

    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)

    selected_name = natural_earth_country_name or country.capitalize()

    if natural_earth_country_column not in gdf.columns:
        available_columns = ", ".join(gdf.columns.astype(str))
        raise ValueError(
            f"Column '{natural_earth_country_column}' was not found. "
            f"Available columns are: {available_columns}"
        )

    matches = gdf[
        gdf[natural_earth_country_column]
        .astype(str)
        .str.lower()
        .eq(str(selected_name).lower())
    ]

    if matches.empty:
        available_examples = ", ".join(
            sorted(gdf[natural_earth_country_column].dropna().astype(str).unique())[:20]
        )
        raise ValueError(
            f"Country '{selected_name}' was not found in column "
            f"'{natural_earth_country_column}'. Examples include: {available_examples}"
        )

    if hasattr(matches.geometry, "union_all"):
        country_geometry = matches.geometry.union_all()
    else:
        country_geometry = matches.geometry.unary_union

    return gpd.GeoSeries([country_geometry], crs="EPSG:4326")
def clip_to_country(ds, country_geometry):
    """Clip an xarray Dataset to the selected country geometry in EPSG:4326."""
    ds = prepare_spatial_dataset(ds)
    return ds.rio.clip(country_geometry, crs="EPSG:4326", all_touched=True)


def merge_dem_tiles(tile_paths, output_path, nodata_value=np.nan, method="first"):
    """Merge multiple GeoTIFF DEM tiles into one GeoTIFF file."""
    tile_paths = [Path(p) for p in tile_paths]
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if len(tile_paths) < 2:
        raise ValueError("At least two tile paths are required for merging.")

    missing = [path for path in tile_paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"The following DEM tiles were not found: {missing}")

    sources = [rasterio.open(path) for path in tile_paths]

    try:
        reference = sources[0]

        for src in sources[1:]:
            if src.crs != reference.crs:
                raise ValueError(f"Input tiles have different CRS: {reference.crs} and {src.crs}")

            if not np.isclose(src.res[0], reference.res[0]) or not np.isclose(src.res[1], reference.res[1]):
                raise ValueError(f"Input tiles have different resolutions: {reference.res} and {src.res}")

        mosaic, transform = merge(sources, nodata=nodata_value, method=method)

        metadata = reference.meta.copy()
        metadata.update(
            {
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": transform,
                "crs": reference.crs,
                "nodata": nodata_value,
                "dtype": "float32",
            }
        )

        with rasterio.open(output_path, "w", **metadata) as dst:
            dst.write(mosaic.astype("float32"))

    finally:
        for src in sources:
            src.close()

    return output_path


def should_write(path, overwrite=True):
    """Return True when an output file should be written."""
    path = Path(path)
    return overwrite or not path.exists()

## Step 4. Processing functions

These functions create the final products. They are kept separate from the run cell so users can inspect the workflow without editing the implementation.

In [4]:
def process_era5_land_orography_and_aspect(
    era5_geopotential_path,
    output_path,
    country,
    country_geometry,
    overwrite=True,
):
    """Create ERA5 Land orography and aspect for the selected country."""
    era5_geopotential_path = Path(era5_geopotential_path)
    output_path = Path(output_path)

    if not era5_geopotential_path.exists():
        raise FileNotFoundError(f"ERA5 Land geopotential file not found: {era5_geopotential_path}")

    global_orography_tif = f"../DEMdata/era5land_orography_global_wgs84.tif"
    global_aspect_tif = f"../DEMdata/era5land_aspect_global_wgs84.tif"

    country_orography_nc = output_path / f"era5land_orography_{country}.nc"
    country_aspect_nc = output_path / f"era5land_aspect_{country}.nc"

    if should_write(global_orography_tif, overwrite):
        geopotential = rioxarray.open_rasterio(era5_geopotential_path)
        orography = geopotential / 9.81
        orography = orography.rio.write_crs("EPSG:4326")
        orography.rio.to_raster(global_orography_tif)

    if should_write(global_aspect_tif, overwrite):
        dem = xdem.DEM(str(global_orography_tif))
        aspect = dem.aspect()
        aspect.save(str(global_aspect_tif))

    if should_write(country_orography_nc, overwrite):
        ds = rioxarray.open_rasterio(global_orography_tif).to_dataset(name="z")
        ds = clip_to_country(ds, country_geometry)
        ds = ds.where(ds != -9999, np.nan)
        ds = drop_auxiliary_coordinates(ds)
        save_netcdf_compressed(ds, country_orography_nc)

    if should_write(country_aspect_nc, overwrite):
        ds = rioxarray.open_rasterio(global_aspect_tif).to_dataset(name="aspect")
        ds = clip_to_country(ds, country_geometry)
        ds = ds.where(ds != -9999, np.nan)
        ds = drop_auxiliary_coordinates(ds)
        save_netcdf_compressed(ds, country_aspect_nc)

    return {
        "era5land_orography": country_orography_nc,
        "era5land_aspect": country_aspect_nc,
    }


def process_glo90_orography_and_aspect(
    glo90_orography_path,
    glo90_tile_paths,
    output_path,
    country,
    country_geometry,
    overwrite=True,
):
    """Create GLO 90 orography and aspect for the selected country."""
    output_path = Path(output_path)
    glo90_orography_path = Path(glo90_orography_path)

    if glo90_tile_paths:
        tile_paths = [Path(p) for p in glo90_tile_paths]
        if should_write(glo90_orography_path, overwrite):
            merge_dem_tiles(tile_paths, glo90_orography_path)

    if not glo90_orography_path.exists():
        raise FileNotFoundError(
            f"GLO 90 orography file not found: {glo90_orography_path}. "
            "Provide glo90_orography_path or define glo90_tile_paths."
        )

    cropped_tif = output_path / f"glo90_orography_{country}_cropped.tif"
    orography_nc = output_path / f"glo90_orography_{country}.nc"
    aspect_tif = output_path / f"glo90_aspect_{country}.tif"
    aspect_nc = output_path / f"glo90_aspect_{country}.nc"

    if should_write(orography_nc, overwrite) or should_write(cropped_tif, overwrite):
        ds = rioxarray.open_rasterio(glo90_orography_path).to_dataset(name="z")
        ds = clip_to_country(ds, country_geometry)
        ds = ds.where(ds != -9999, np.nan)
        ds = ds.rio.write_crs("EPSG:4326")
        ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")

        if should_write(cropped_tif, overwrite):
            ds.rio.to_raster(cropped_tif)

        if should_write(orography_nc, overwrite):
            save_netcdf_compressed(ds, orography_nc)

    if should_write(aspect_tif, overwrite):
        dem_source = cropped_tif if cropped_tif.exists() else glo90_orography_path
        dem = xdem.DEM(str(dem_source))
        aspect = dem.aspect()
        aspect.save(str(aspect_tif))

    if should_write(aspect_nc, overwrite):
        aspect_ds = xr.open_dataset(aspect_tif)
        aspect_ds = rename_coordinates(aspect_ds)

        if "band_data" in aspect_ds:
            aspect_ds = aspect_ds.rename({"band_data": "aspect"})

        if "band" in aspect_ds.dims:
            aspect_ds = aspect_ds.sel(band=1).drop_vars("band")

        aspect_ds = aspect_ds.rio.write_crs("EPSG:4326")
        aspect_ds = drop_auxiliary_coordinates(aspect_ds)
        save_netcdf_compressed(aspect_ds, aspect_nc)

    return {
        "glo90_orography": orography_nc,
        "glo90_aspect": aspect_nc,
    }

## Step 5. Run the workflow

Run this cell after setting the parameters.

The crop is performed using `country_geometry`. If `boundary_source = "local_dissolve"`, all polygons in the local shapefile are merged into one country geometry first. This is the recommended option for `REGIONES_v1.shp`.

The final files are written in:

```python
./DEMdata/{country}/
```


In [ ]:
country_geometry = load_country_geometry(
    country=country,
    #boundary_source=boundary_source,
    #local_boundary_column=local_boundary_column,
    #local_boundary_values=local_boundary_values,
    natural_earth_path=natural_earth_path,
    natural_earth_country_column=natural_earth_country_column,
    natural_earth_country_name=natural_earth_country_name,
)

print("Country geometry loaded successfully.")
print(f"Number of geometries used for clipping: {len(country_geometry)}")
print(f"Geometry bounds in EPSG:4326: {country_geometry.total_bounds}")

outputs = {}

if run_era5_land:
    outputs.update(
        process_era5_land_orography_and_aspect(
            era5_geopotential_path=era5_geopotential_path,
            output_path=output_path,
            country=country,
            country_geometry=country_geometry,
            overwrite=overwrite,
        )
    )

if run_glo90:
    outputs.update(
        process_glo90_orography_and_aspect(
            glo90_orography_path=glo90_orography_path,
            glo90_tile_paths=glo90_tile_paths,
            output_path=output_path,
            country=country,
            country_geometry=country_geometry,
            overwrite=overwrite,
        )
    )

print("Processing completed. Generated files:")
for name, path in outputs.items():
    print(f"{name}: {path}")

## Notes for users
If you use a local shapefile but only want a subset of regions, set `boundary_source = "local_filter"`, then define `local_boundary_column` and `local_boundary_values`.

If you want to use the Natural Earth global shapefile instead, set `boundary_source = "natural_earth"` and check that `natural_earth_country_column` and `natural_earth_country_name` match the file.

If the GLO 90 DEM is split into several files, place all tiles in `./DEMdata/{country}/` and list them in `glo90_tile_paths`.

The generated NetCDF files are the files expected by the downstream ATLAS notebooks that use orography and aspect as predictors.
